In [ ]:
'''# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session'''

In [1]:
import numpy as np
import pandas as pd
import pickle
#df=pd.read_csv("/kaggle/input/datasets/adithip2000/genre-generalised-and-tagged/final_processed_data.csv")


In [2]:
#loading test data
with open("/kaggle/input/datasets/mrxn251/nlp-project-others/splits.pkl","rb") as f:
    data=pickle.load(f)
X_test=data["X_test"]
y_test=data["y_test"]

In [2]:
#!pip install iterative-stratification

In [3]:
#df.drop(columns=['Unnamed: 0'],inplace=True)

In [4]:
#df.head()

,genres,form,summary
0,"['sci-fi', 'drama', 'family']",book,"old major, the old boar on the manor farm, cal..."
1,"['sci-fi', 'drama']",book,"alex, a teenager living in near-future england..."
2,['drama'],book,the text of the plague is divided into five pa...
3,"['sci-fi', 'fantasy', 'drama']",book,the novel posits that space around the milky w...
4,"['sci-fi', 'fantasy', 'drama', 'family']",book,"ged is a young boy on gont, one of the larger ..."


In [7]:
from transformers import AutoTokenizer,AutoModel
import torch
model_id="roberta-base"
tokenizer=AutoTokenizer.from_pretrained(model_id)
#model = AutoModel.from_pretrained(model_id)
#device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
#model.to(device)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
'''import ast
type(df['genres'].iloc[0])
df['genres']=df['genres'].apply(ast.literal_eval)

type(df['genres'].iloc[0])'''

list

In [6]:
'''from sklearn.preprocessing import MultiLabelBinarizer
mlb=MultiLabelBinarizer()
y=mlb.fit_transform(df['genres'])
print(y[0])'''

#Load binarizer
with open("/kaggle/input/datasets/mrxn251/nlp-project-others/mlb.pkl","rb") as f:
    mlb=pickle.load(f)

In [8]:
'''from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

X = df['summary'].reset_index(drop=True)
y = y  

# 🔹 First split: train vs temp (80/20)
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, temp_idx in msss.split(X, y):
    X_train, X_temp = X.iloc[train_idx], X.iloc[temp_idx]
    y_train, y_temp = y[train_idx], y[temp_idx]

# 🔹 Second split: val vs test (10/10)
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)

for val_idx, test_idx in msss2.split(X_temp, y_temp):
    X_val, X_test = X_temp.iloc[val_idx], X_temp.iloc[test_idx]
    y_val, y_test = y_temp[val_idx], y_temp[test_idx]
'''

In [8]:
import torch
from torch.utils.data import Dataset

class GenreDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, vectorizer, max_len=128):
        """
        texts: pandas Series of text
        labels: multi-hot numpy array or list
        tokenizer: HuggingFace tokenizer
        vectorizer: fitted CountVectorizer / TfidfVectorizer
        """

        self.texts = texts.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

        
        self.labels = torch.tensor(labels, dtype=torch.float32)

        
        bow_matrix = vectorizer.transform(self.texts)
        self.bow = torch.tensor(bow_matrix.toarray(), dtype=torch.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        #  Tokenize
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        #  Remove batch dimension
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "bow": self.bow[idx],
            "labels": self.labels[idx]
        }

In [9]:
'''from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(
    max_features=8000,
    stop_words='english',
    min_df=5,
    max_df=0.8,
    ngram_range=(1,1)  # keep simple for now
)
vectorizer.fit(X_train.values)
vectorizer.transform(X_test.values)'''

#Load vectorizer
with open("/kaggle/input/datasets/mrxn251/nlp-project-others/vectorizer.pkl","rb") as f:
    vectorizer=pickle.load(f)


In [12]:
import torch
import torch.nn as nn
from transformers import AutoModel

class TopicGenreModel(nn.Module):
    def __init__(self, num_topics, num_genres, vocab_size, model_name="roberta-base"):
        super().__init__()

        # 🔹 Backbone (RoBERTa)
        self.backbone = AutoModel.from_pretrained(model_name)
        bert_dim = self.backbone.config.hidden_size

        # 🔹 Context encoder ONLY (no BoW encoder)
        self.encoder = nn.Sequential(
            nn.Linear(bert_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # 🔹 Latent space
        self.fc_mu = nn.Linear(256, num_topics)
        self.fc_logvar = nn.Linear(256, num_topics)

        # 🔹 Topic → word matrix
        self.beta = nn.Parameter(torch.randn(num_topics, vocab_size))
        nn.init.xavier_uniform_(self.beta)

        # 🔹 Topic → genre classifier
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_topics, num_genres)
        )

    # ---------- Encode ----------
    def encode(self, embedding):
        h = self.encoder(embedding)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    # ---------- Reparameterization ----------
    def reparameterize(self, mu, logvar):
        logvar = torch.clamp(logvar, -10, 10)  # stability
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    # ---------- Decode ----------
    def decode(self, topic_dist):
       # beta_norm = torch.softmax(self.beta, dim=1)  
        logits = torch.matmul(topic_dist, self.beta)/0.5
        return torch.log_softmax(logits, dim=1)

    # ---------- Forward ----------
    def forward(self, input_ids, attention_mask):

        # 🔹 Get contextual embedding (MEAN pooling)
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)

        mask = attention_mask.unsqueeze(-1)
        embedding = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)

        # 🔹 Encode → latent topics
        mu, logvar = self.encode(embedding)

        # 🔹 Sample
        z = self.reparameterize(mu, logvar)
        #z = mu + 0.1 * torch.randn_like(mu)
        topic_dist = torch.softmax(z/0.5, dim=1)

        # 🔹 Genre prediction
        genre_logits = self.classifier(topic_dist)

        # 🔹 Word reconstruction
        word_log_probs = self.decode(topic_dist)

        return topic_dist, genre_logits, word_log_probs, mu, logvar

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_genres = len(mlb.classes_)
num_topics = 32
vocab_size=8000
print(f"Vocab size: {vocab_size}")
print(f"num_genres: {num_genres}")
print(f"num_topics: {num_topics}")

model = TopicGenreModel(
    num_topics=num_topics,
    num_genres=num_genres,
    vocab_size=vocab_size,
    model_name="roberta-base"
)

model.load_state_dict(torch.load("/kaggle/input/datasets/mrxn251/roberta-ver2/checkpoint.pt",map_location=device))
model.to(device)
model.eval()

Vocab size: 8000
num_genres: 13
num_topics: 32


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TopicGenreModel(
  (backbone): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): La

In [15]:
test_dataset = GenreDataset(
    texts=X_test,
    labels=y_test,
    tokenizer=tokenizer,
    vectorizer=vectorizer,
    max_len=128
)


In [16]:
sample = test_dataset[0]

print(sample["input_ids"].shape)     # [256]
print(sample["attention_mask"].shape)
print(sample["bow"].shape)           # [8000]
print(sample["labels"].shape)        # [num_genres]



torch.Size([128])
torch.Size([128])
torch.Size([8000])
torch.Size([13])


In [17]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


In [18]:
import torch
from torchmetrics.classification import (
    MultilabelAccuracy,
    MultilabelF1Score,
    MultilabelPrecision,
    MultilabelRecall
)

def test_model(model, loader, device, num_labels, mlb, threshold=0.3):

    model.eval()

    # 🔹 Overall metrics
    acc_metric = MultilabelAccuracy(num_labels=num_labels, threshold=threshold).to(device)
    f1_macro = MultilabelF1Score(num_labels=num_labels, average='macro', threshold=threshold).to(device)
    f1_weighted = MultilabelF1Score(num_labels=num_labels, average='weighted', threshold=threshold).to(device)

    # 🔹 Per-class metrics
    f1_per_class = MultilabelF1Score(num_labels=num_labels, average=None, threshold=threshold).to(device)
    precision_per_class = MultilabelPrecision(num_labels=num_labels, average=None, threshold=threshold).to(device)
    recall_per_class = MultilabelRecall(num_labels=num_labels, average=None, threshold=threshold).to(device)

    # 🔹 Reset all
    acc_metric.reset()
    f1_macro.reset()
    f1_weighted.reset()
    f1_per_class.reset()
    precision_per_class.reset()
    recall_per_class.reset()

    all_preds = []
    all_labels = []
    all_topics = []

    with torch.no_grad():
        for batch in loader:

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # 🔹 Forward
            topic_dist, genre_logits, _, _, _ = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            probs = torch.sigmoid(genre_logits)

            # 🔹 Update metrics
            acc_metric.update(probs, labels.int())
            f1_macro.update(probs, labels.int())
            f1_weighted.update(probs, labels.int())

            f1_per_class.update(probs, labels.int())
            precision_per_class.update(probs, labels.int())
            recall_per_class.update(probs, labels.int())

            # 🔹 Store
            all_preds.append(probs.cpu())
            all_labels.append(labels.cpu())
            all_topics.append(topic_dist.cpu())

    # 🔹 Compute overall
    print("\n=== Overall Metrics ===")
    print("Accuracy:", acc_metric.compute().item())
    print("Macro F1:", f1_macro.compute().item())
    print("Weighted F1:", f1_weighted.compute().item())

    # 🔹 Compute per-class
    f1_vals = f1_per_class.compute().cpu().numpy()
    precision_vals = precision_per_class.compute().cpu().numpy()
    recall_vals = recall_per_class.compute().cpu().numpy()

    print("\n=== Per-Class Metrics ===")
    for i in range(num_labels):
        print(f"{mlb.classes_[i]:<12} | F1: {f1_vals[i]:.4f} | Precision: {precision_vals[i]:.4f} | Recall: {recall_vals[i]:.4f}")

    return (
        torch.cat(all_preds),
        torch.cat(all_labels),
        torch.cat(all_topics)
    )

In [21]:
def explain_prediction(probs, topics, model, vectorizer, mlb, threshold=0.3, top_topics=3, top_words=6):

    beta = model.beta.detach().cpu().numpy()
    vocab = vectorizer.get_feature_names_out()
    weights = model.classifier[1].weight.detach().cpu().numpy()

    for i in range(len(probs)):

        pred = (probs[i] > threshold).int()

        print(f"\n Sample {i}")

        for genre_idx, val in enumerate(pred):
            if val == 1:

                genre = mlb.classes_[genre_idx]
                print(f"\n Genre: {genre}")

                topic_contrib = topics[i].numpy() * weights[genre_idx]
                top_topic_ids = topic_contrib.argsort()[-top_topics:][::-1]

                for t in top_topic_ids:
                    word_ids = beta[t].argsort()[-top_words:][::-1]
                    words = [vocab[j] for j in word_ids]

                    print(f"  Topic {t}: {words}")

In [23]:
preds, labels, topics = test_model(
    model,
    test_loader,
    device,
    num_genres,
    mlb,
    threshold=0.5
)

# Explain first few samples
explain_prediction(preds[:5], topics[:5], model, vectorizer, mlb)


=== Overall Metrics ===
Accuracy: 0.6221030950546265
Macro F1: 0.3265375792980194
Weighted F1: 0.4216601848602295

=== Per-Class Metrics ===
action       | F1: 0.2524 | Precision: 0.1680 | Recall: 0.5074
adventure    | F1: 0.2140 | Precision: 0.1238 | Recall: 0.7917
comedy       | F1: 0.3896 | Precision: 0.2888 | Recall: 0.5983
crime        | F1: 0.2579 | Precision: 0.1529 | Recall: 0.8221
drama        | F1: 0.6397 | Precision: 0.6772 | Recall: 0.6061
family       | F1: 0.3780 | Precision: 0.2510 | Recall: 0.7645
fantasy      | F1: 0.3370 | Precision: 0.2115 | Recall: 0.8284
history      | F1: 0.0790 | Precision: 0.0419 | Recall: 0.6911
horror       | F1: 0.2537 | Precision: 0.1518 | Recall: 0.7722
mystery      | F1: 0.1988 | Precision: 0.1186 | Recall: 0.6160
romance      | F1: 0.4253 | Precision: 0.2994 | Recall: 0.7343
sci-fi       | F1: 0.4806 | Precision: 0.3320 | Recall: 0.8700
thriller     | F1: 0.3389 | Precision: 0.2386 | Recall: 0.5851

 Sample 0

 Genre: action
  Topic 2: [

In [24]:

import torch
import numpy as np

def predict_and_explain(text, model, tokenizer, vectorizer, mlb, device,
                        threshold=0.5, top_topics=3, top_words=8):

    model.eval()

    # 🔹 Tokenize input
    encoding = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # 🔹 Forward pass
    with torch.no_grad():
        topic_dist, genre_logits, _, _, _ = model(input_ids, attention_mask)

        probs = torch.sigmoid(genre_logits).cpu().numpy()[0]
        topic_dist = topic_dist.cpu().numpy()[0]

    # 🔹 Get predicted labels
    pred_binary = (probs > threshold).astype(int)
    predicted_genres = mlb.inverse_transform(pred_binary.reshape(1, -1))[0]

    # 🔹 Load model components
    beta = model.beta.detach().cpu().numpy()
    vocab = vectorizer.get_feature_names_out()
    classifier_weights = model.classifier[1].weight.detach().cpu().numpy()

    print("\n📌 Input Summary:")
    print(text)

    print("\n🎯 Predicted Genres:")
    print(predicted_genres if len(predicted_genres) > 0 else "None")

    print("\n📊 Probabilities:")
    for i, g in enumerate(mlb.classes_):
        print(f"{g}: {probs[i]:.3f}")

    # 🔹 Explain per genre
    for genre in predicted_genres:

        genre_idx = list(mlb.classes_).index(genre)

        print(f"\n🔍 Explanation for Genre: {genre}")

        # topic contribution = θ * classifier weight
        topic_contrib = topic_dist * classifier_weights[genre_idx]

        top_topic_ids = topic_contrib.argsort()[-top_topics:][::-1]

        for t in top_topic_ids:

            word_ids = beta[t].argsort()[-top_words:][::-1]
            words = [vocab[j] for j in word_ids]

            print(f" Topic {t}: {words}")

In [27]:
#text="Death on the Nile is a classic 1937 Agatha Christie murder mystery featuring Hercule Poirot on a luxurious Nile cruise. Heiress Linnet Doyle is murdered after being stalked by her friend Jackie, whose fiancé, Simon, Linnet stole. Poirot reveals a conspiracy: Simon and Jackie conspired to kill Linnet for her fortun"
text="Dhurandhar (2025) is a high-octane Bollywood spy thriller directed by Aditya Dhar and starring Ranveer Singh as Hamza Ali Mazari, an undercover Indian intelligence agent. The film follows his dangerous mission to infiltrate Karachi's violent Lyari underworld to dismantle the Pakistan-based terror network following the 2001 Parliament attacks"
predict_and_explain(text,model,tokenizer,vectorizer,mlb,device,threshold=0.5)


📌 Input Summary:
Dhurandhar (2025) is a high-octane Bollywood spy thriller directed by Aditya Dhar and starring Ranveer Singh as Hamza Ali Mazari, an undercover Indian intelligence agent. The film follows his dangerous mission to infiltrate Karachi's violent Lyari underworld to dismantle the Pakistan-based terror network following the 2001 Parliament attacks

🎯 Predicted Genres:
('action', 'crime', 'drama', 'mystery', 'thriller')

📊 Probabilities:
action: 0.522
adventure: 0.452
comedy: 0.495
crime: 0.536
drama: 0.506
family: 0.439
fantasy: 0.337
history: 0.415
horror: 0.483
mystery: 0.537
romance: 0.454
sci-fi: 0.339
thriller: 0.552

🔍 Explanation for Genre: action
 Topic 25: ['man', 'film', 'new', 'time', 'life', 'story', 'father', 'young']
 Topic 0: ['police', 'man', 'death', 'young', 'film', 'new', 'finds', 'wife']
 Topic 15: ['man', 'film', 'young', 'new', 'police', 'time', 'death', 'father']

🔍 Explanation for Genre: crime
 Topic 7: ['love', 'film', 'life', 'family', 'father', 'm